In [ ]:
!pip install -U bitsandbytes datasets transformers peft trl accelerate huggingface_hub

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
import pandas as pd
import time
import gc
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from huggingface_hub import login

# ==========================================
# 0. HUGGING FACE GİRİŞİ & VERİ HAZIRLIĞI
# ==========================================
login(token="XXXXXXXXXX")

train_path = "drive/MyDrive/LLM-PROJE/augmented_train_5000.csv"
metrics_csv_path = "drive/MyDrive/LLM-PROJE/fine_tuning_metrics.csv"
loss_log_path = "drive/MyDrive/LLM-PROJE/fine_tuning_loss_logs.csv"

df = pd.read_csv(train_path)

# Verinin tamamını garantiye almak için fonksiyonu güncelledik
def format_prompts(examples):
    formatted_texts = []
    for text, label in zip(examples['İcerik'], examples['Kategori']):
        formatted_text = f"<bos><start_of_turn>user\nTalimat: Sen bir İş Hukuku uzmanısın. Aşağıdaki içeriği oku ve sadece ilgili kategorisini tek kelimeyle yaz. Açıklama yapma.\n\nİçerik: {text}<end_of_turn>\n<start_of_turn>model\n{label}<end_of_turn><eos>"
        formatted_texts.append(formatted_text)
    return {"text": formatted_texts}

raw_dataset = Dataset.from_pandas(df)
# Verilerin haritalanması iyileştirildi
dataset = raw_dataset.map(format_prompts, batched=True)

# ==========================================
# 1. MODEL LİSTESİ
# ==========================================

MODELS = {
    "Gemma-2B": "google/gemma-2-2b-it",
    "Qwen-2.5-1.5B": "Qwen/Qwen2.5-1.5B-Instruct",
    "SmolLM2-360M": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "TinyLlama-1.1B": "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
}
SEEDS = [42, 123, 999]

# Mevcut csv dosyaları varsa üstüne yazmamak, korumak için kontrol
if os.path.exists(metrics_csv_path):
    try: metrics_results = pd.read_csv(metrics_csv_path).to_dict(orient="records")
    except: metrics_results = []
else:
    metrics_results = []

if os.path.exists(loss_log_path):
    try: loss_logs = pd.read_csv(loss_log_path).to_dict(orient="records")
    except: loss_logs = []
else:
    loss_logs = []

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# ==========================================
# 2. ANA OTOMASYON DÖNGÜSÜ
# ==========================================
for model_name, model_id in MODELS.items():
    # NOT: Eğer daha önce eğitilen Gemma, Qwen ve TinyLlama'yı tekrar eğitmek istemiyorsan,
    # sadece SmolLM2 eksik kaldığı için buradaki kontrolü kullanabilirsin.
    # Biz şimdi her şeyi tam kapasite (tüm adımlarla) görmek için döngüyü çalıştırıyoruz.

    for seed in SEEDS:
        # Eğer bu model ve seed kombinasyonu zaten başarıyla listede varsa atla
        if any(m["Model"] == model_name and m["Seed"] == seed for m in metrics_results if "Egitim_Suresi_Saniye" in m):
            print(f"[ATLANDI] {model_name} (Seed {seed}) zaten eğitilmiş.")
            continue

        print("\n" + "="*50)
        print(f"[BAŞLADI] Model: {model_name} | Seed: {seed} Gerçek Eğitimi Başlıyor...")
        print("="*50)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        set_seed(seed)
        torch.manual_seed(seed)

        try:
            tokenizer = AutoTokenizer.from_pretrained(model_id)
            tokenizer.pad_token = tokenizer.eos_token
            tokenizer.padding_side = "right"

            # Verinin 5100 satırını da tam olarak işleyen tokenizasyon adımı
            def tokenize_function(examples):
                return tokenizer(examples["text"], truncation=True, max_length=512)

            tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

            base_model = AutoModelForCausalLM.from_pretrained(
                model_id,
                quantization_config=bnb_config,
                device_map="auto"
            )
            base_model = prepare_model_for_kbit_training(base_model)

            peft_config = LoraConfig(
                r=8,
                lora_alpha=16,
                target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
                lora_dropout=0.05,
                bias="none",
                task_type="CAUSAL_LM"
            )
            model = get_peft_model(base_model, peft_config)

            # Gerçek eğitim loglarını görebilmek için max_steps sınırını kaldırdık, normal akışa bıraktık
            training_args = TrainingArguments(
                output_dir=f"./results_{model_name}_{seed}",
                num_train_epochs=1,
                per_device_train_batch_size=4,
                gradient_accumulation_steps=4,       # VRAM'i korumak için 4'e çektik
                optim="paged_adamw_8bit",
                logging_steps=10,                    # Her 10 adımda bir ekrana Loss basacak
                learning_rate=2e-4,
                fp16=False,
                bf16=False,
                gradient_checkpointing=True,
                gradient_checkpointing_kwargs={"use_reentrant": False},
                save_strategy="no",
                seed=seed
            )

            data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

            trainer = Trainer(
                model=model,
                train_dataset=tokenized_dataset,
                args=training_args,
                data_collator=data_collator
            )

            start_time = time.time()
            train_result = trainer.train()
            end_time = time.time()

            execution_time = end_time - start_time
            peak_vram = torch.cuda.max_memory_allocated() / (1024 ** 3) if torch.cuda.is_available() else 0

            print(f"\n[TAMAMLANDI] Süre: {execution_time:.2f} sn | Zirve VRAM: {peak_vram:.2f} GB")

            metrics_results.append({
                "Model": model_name,
                "Seed": seed,
                "Egitim_Suresi_Saniye": execution_time,
                "Zirve_VRAM_GB": peak_vram,
                "Final_Loss": train_result.training_loss
            })

            for log in trainer.state.log_history:
                if "loss" in log:
                    loss_logs.append({
                        "Model": model_name,
                        "Seed": seed,
                        "Step": log["step"],
                        "Loss": log["loss"]
                    })

            # Gelişmeleri anlık olarak CSV'ye kaydet
            pd.DataFrame(metrics_results).to_csv(metrics_csv_path, index=False, encoding='utf-8-sig')
            pd.DataFrame(loss_logs).to_csv(loss_log_path, index=False, encoding='utf-8-sig')

            # Modeli kaydetme
            save_dir = f"drive/MyDrive/LLM-PROJE/models/{model_name}_seed_{seed}"
            trainer.model.save_pretrained(save_dir)
            tokenizer.save_pretrained(save_dir)

        except Exception as e:
            print(f"\n[HATA] {model_name} (Seed {seed}) atlandı: {e}")

        if 'trainer' in locals(): del trainer
        if 'model' in locals(): del model
        if 'base_model' in locals(): del base_model
        if 'tokenizer' in locals(): del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n" + "="*50)
print(f"[EKSİKSİZ TAMAMLANDI] Eksik kalan tüm turlar veri setinin tamamıyla eğitildi!")
print("="*50)